# 에이전트 (Agent)

에이전트는 언어 모델(LLM)과 도구(Tool)를 결합하여 복잡한 작업을 수행하는 시스템입니다. 에이전트는 주어진 작업에 대해 추론하고, 필요한 도구를 선택하며, 목표를 향해 반복적으로 작업을 수행합니다.

![](./assets/langgraph-agent.png)

LangChain의 `create_agent` 함수는 프로덕션 수준의 에이전트 구현을 제공합니다. 이 함수를 사용하면 모델 선택, 도구 연동, 미들웨어 설정 등을 손쉽게 구성할 수 있습니다.

> 참고 문서: [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)

## 환경 설정

에이전트 튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로깅 기능을 활성화하여 LangSmith에서 실행 추적을 확인할 수 있도록 합니다.

LangSmith 추적을 활성화하면 에이전트의 추론 과정, 도구 호출, 응답 생성 등을 시각적으로 디버깅할 수 있어 개발에 큰 도움이 됩니다.

아래 코드는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)
# 추적을 위한 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangGraph-V1-Tutorial


## 모델 (Model)

에이전트의 추론 엔진인 LLM은 `create_agent` 함수의 첫 번째 인자로 전달합니다. 모델을 지정하는 방법은 크게 두 가지가 있습니다.

### 방법 1: 문자열 식별자 (provider:model)

가장 간단한 방법은 `provider:model` 형식의 문자열을 직접 `create_agent`에 전달하는 것입니다. 이 방식은 빠른 프로토타이핑에 적합합니다.

### 방법 2: init_chat_model 함수

`init_chat_model` 함수를 사용하면 모델 인스턴스를 먼저 생성한 후 전달할 수 있습니다. 이 방식은 모델 옵션을 세밀하게 제어할 때 유용합니다.

아래 코드는 두 가지 방법을 모두 보여주는 예시입니다.

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 방법 1: 문자열 식별자를 직접 전달 (빠른 프로토타이핑에 적합)
# agent = create_agent("anthropic:claude-sonnet-4-5", tools=[])

# 방법 2: init_chat_model로 모델 인스턴스 생성 후 전달
# OpenAI 키를 사용하는 경우 gpt-4.1-mini, gpt-4.1 등으로 변경하세요
model = init_chat_model("claude-sonnet-4-5")

# 모델 인스턴스를 사용하여 에이전트 생성
agent = create_agent(model, tools=[])

### 모델 세부 설정

더 세밀한 제어가 필요한 경우, 모델 생성 시 추가 옵션을 전달하여 다양한 설정을 적용할 수 있습니다.

- `temperature`: 응답의 무작위성을 제어합니다 (0에 가까울수록 일관된 응답, 1에 가까울수록 다양한 응답)
- `max_tokens`: 생성할 최대 토큰 수를 제한합니다
- `timeout`: 요청 타임아웃(초)을 설정합니다

모델 세부 설정을 적용하는 방법은 두 가지가 있습니다.

**방법 1: init_chat_model 사용**

`init_chat_model` 함수에 추가 옵션을 전달하는 방식입니다. Provider에 관계없이 동일한 인터페이스로 사용할 수 있어 편리합니다.

**방법 2: Provider별 클래스 직접 사용**

`ChatAnthropic`, `ChatOpenAI` 등 Provider별 클래스를 직접 인스턴스화하는 방식입니다. Provider 고유의 옵션을 세밀하게 제어할 때 유용합니다.

아래 코드는 두 가지 방법을 모두 보여주는 예시입니다.

In [4]:
# 방법 1: init_chat_model에 추가 옵션 전달
# Provider에 관계없이 동일한 인터페이스로 사용 가능
model = init_chat_model(
    "claude-sonnet-4-5",  # OpenAI 키를 사용하는 경우 gpt-4.1-mini, gpt-4.1 등으로 변경하세요
    temperature=0.1,  # 응답의 무작위성 제어
    max_tokens=1000,  # 최대 생성 토큰 수
    timeout=30,  # 요청 타임아웃(초)
)

agent = create_agent(model, tools=[])

In [5]:
# 방법 2: Provider별 클래스 직접 사용
# Provider 고유의 옵션을 세밀하게 제어할 때 유용
from langchain_anthropic import ChatAnthropic
# from langchain_openai import ChatOpenAI  # OpenAI 사용 시

model = ChatAnthropic(
    model="claude-sonnet-4-5",
    temperature=0.1,
    max_tokens=1000,
    timeout=30,
)

agent = create_agent(model, tools=[])

### 동적 모델 선택

동적 모델 선택은 런타임에 현재 상태와 컨텍스트를 기반으로 사용할 모델을 결정하는 패턴입니다. 이를 통해 정교한 라우팅 로직과 비용 최적화가 가능합니다. 예를 들어, 간단한 질문에는 경량 모델을, 복잡한 대화에는 고급 모델을 사용할 수 있습니다.

`wrap_model_call` 데코레이터를 사용하면 모델 호출 전에 요청을 검사하고 수정할 수 있는 미들웨어를 생성할 수 있습니다.

![](../assets/wrap_model_call.png)

아래 코드는 대화 길이에 따라 모델을 동적으로 선택하는 예시입니다.

### ModelRequest 속성

`ModelRequest`는 에이전트의 모델 호출 정보를 담는 데이터 클래스로, 미들웨어에서 요청을 검사하고 수정할 때 사용됩니다. `override()` 메서드를 통해 여러 속성을 동시에 변경할 수 있습니다.

아래 코드는 ModelRequest를 사용하여 동적으로 모델과 시스템 프롬프트를 변경하는 예시입니다.

In [10]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

# 기본 모델과 고급 모델 정의
basic_model = init_chat_model("claude-haiku-4-5")
advanced_model = init_chat_model("claude-sonnet-4-5")


@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """대화 복잡도에 따라 모델 선택"""
    message_count = len(request.state["messages"])

    # 긴 대화에는 고급 모델 사용
    if message_count > 10:
        model = advanced_model
    else:
        model = basic_model

    request.model = model
    print(f"모델 선택: {request.model.model}")
    return handler(request)


agent = create_agent(
    model=basic_model, tools=[], middleware=[dynamic_model_selection]  # 기본 모델
)

In [13]:
from langchain_teddynote.messages import stream_graph
from langchain_core.messages import HumanMessage

stream_graph(
    agent,
    inputs={
        "messages": [HumanMessage(content="머신러닝의 동작 원리에 대해서 설명해줘")]
    },
)

/var/folders/v7/jvdq74bd213fzs8k9cmpl_7w0000gn/T/ipykernel_63447/2629723978.py:19: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


모델 선택: claude-haiku-4-5

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
# 머신러닝의 동작 원리

머신러닝은 **데이터로부터 패턴을 학습하여 예측이나 판단을 하는 기술**입니다. 기본 동작 과정을 설명하겠습니다.

## 🎯 기본 개념

```
데이터 입력 → 패턴 학습 → 모델 생성 → 예측/판단
```

## 1️⃣ **학습 데이터 수집**
- 충분한 양의 데이터가 필요합니다
- 예: 고양이 사진 10,000장

## 2️⃣ **특징(Feature) 추출**
- 데이터에서 중요한 정보를 뽑아냅니다
- 예: 사진의 색상, 모양, 질감 등

## 3️⃣ **모델 학습 (가중치 조정)**

```
초기값 → 계산 → 오류 측정 → 가중치 조정 
                     ↑
                  반복
```

- **손실함수**: 예측값과 실제값의 차이를 계산
- **경사하강법**: 오류를 줄이는 방향으로 계속 조정

## 4️⃣ **성능 평가**
- 학습하지 않은 테스트 데이터로 검증
- 정확도, 재현율 등으로 평가

## 5️⃣ **예측**
- 학습된 모델이 새로운 데이터를 분류/예측

---

## 📊 머신러닝의 주요 유형

| 유형 | 설명 | 예시 |
|------|------|------|
| **지도학습** | 정답을 알려주며 학습 | 스팸 메일 분류 |
| **비지도학습** | 정답 없이 패턴을 찾음 | 고객 분류 |
| **강화학습** | 보상으로 학습 | 게임 AI |

---

## 💡 쉬운 비유

**아이가 동물을 배우는 과정:**
1. 많은 고양이 사진을 본다 (데이터)
2. "귀가 있고, 수염이 있고..." 특징을 인식 (특징 추출)
3. 규칙을 만들어간다 (모델 학습)
4. 새로운 고양이 사진을 보고 "고양이다!" 판단 (예측)

궁금한 부분이 있으면 더 자세히 설명해드릴 수 있습니다! 😊

**ModelRequest 주요 속성:**

| 속성 | 설명 |
|:---|:---|
| `model` | 사용할 `BaseChatModel` 인스턴스 |
| `system_prompt` | 시스템 프롬프트 (선택적) |
| `messages` | 대화 메시지 리스트 (시스템 프롬프트 제외) |
| `tool_choice` | 도구 선택 설정 |
| `tools` | 사용 가능한 도구 리스트 |
| `response_format` | 응답 형식 지정 |
| `state` | 현재 에이전트 상태 (`AgentState`) |
| `runtime` | 에이전트 런타임 정보 |
| `model_settings` | 추가 모델 설정 (dict) |

아래 코드는 `override()` 메서드를 사용하여 여러 속성을 동시에 변경하는 예시입니다.

In [14]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """대화 복잡도에 따라 모델 선택"""
    message_count = len(request.state["messages"][-1].content)
    print(f"글자수: {message_count}")

    # 긴 대화에는 고급 모델 사용
    if message_count > 10:
        # 여러 속성 동시 변경
        new_request = request.override(
            model=advanced_model,
            system_prompt="emoji 를 사용해서 답변해줘",
            tool_choice="auto",
        )
    else:
        new_request = request.override(
            model=basic_model,
            system_prompt="한 문장으로 간결하게 답변해줘. emoji 는 사용하지 말아줘.",
            tool_choice="auto",
        )
    print(f"모델 선택: {new_request.model.model}")
    return handler(new_request)

    
agent = create_agent(
    model=basic_model, tools=[], middleware=[dynamic_model_selection]  # 기본 모델
)

### 글자수 기반 모델 선택 테스트

아래는 글자수 10자 미만일 때의 응답입니다. 간결한 답변을 생성하도록 설정되어 있습니다.

In [15]:
stream_graph(agent, inputs={"messages": [HumanMessage(content="머신러닝 동작원리")]})

글자수: 9
모델 선택: claude-haiku-4-5

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
# 머신러닝 동작원리

머신러닝은 프로그래밍된 규칙 대신 데이터로부터 패턴을 학습하여 예측이나 분류를 수행하는 기술로, 크게 데이터 수집 → 특성 추출 → 모델 선택 → 학습(가중치 조정) → 예측 단계를 거쳐 작동합니다.

아래는 글자수 10자 이상일 때의 응답입니다. 이모지를 사용하여 친근한 답변을 생성하도록 설정되어 있습니다.

In [16]:
stream_graph(
    agent,
    inputs={
        "messages": [
            HumanMessage(content="머신러닝의 동작 원리에 대해서 설명해 주세요.")
        ]
    },
)

글자수: 25
모델 선택: claude-sonnet-4-5

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
# 머신러닝의 동작 원리 🤖

머신러닝은 컴퓨터가 데이터로부터 스스로 학습하는 기술이에요! 🎯

## 기본 동작 과정 🔄

### 1️⃣ 데이터 수집 📊
- 학습에 필요한 데이터를 모아요
- 예: 고양이 사진 1000장 🐱

### 2️⃣ 모델 선택 🏗️
- 문제에 맞는 알고리즘을 선택해요
- 신경망, 결정트리, SVM 등

### 3️⃣ 학습 과정 📚
```
입력 데이터 → 모델 → 예측 결과
              ↓
         실제 답과 비교
              ↓
         오차 계산 📉
              ↓
         모델 파라미터 조정 🔧
```

### 4️⃣ 반복 학습 🔁
- 오차가 줄어들 때까지 반복해요
- 점점 정확도가 높아져요! ⬆️

### 5️⃣ 평가 및 예측 ✅
- 새로운 데이터로 성능 테스트
- 실전에 적용! 🚀

## 핵심 원리 💡

**"경험(데이터)을 통해 성능을 개선하는 것"**

마치 아기가 넘어지면서 걷기를 배우는 것처럼, 컴퓨터도 실수를 반복하며 배워요! 👶➡️🚶

---

## 프롬프트

에이전트의 동작을 제어하는 핵심 요소 중 하나는 시스템 프롬프트입니다. 시스템 프롬프트를 통해 에이전트의 역할, 응답 스타일, 제약 조건 등을 정의할 수 있습니다.

### 시스템 프롬프트

`system_prompt` 매개변수를 사용하여 에이전트의 기본 동작을 정의할 수 있습니다. 시스템 프롬프트는 모든 대화에서 일관되게 적용되며, 에이전트의 페르소나와 응답 가이드라인을 설정하는 데 사용됩니다.

아래 코드는 간결하고 정확한 응답을 생성하도록 시스템 프롬프트를 설정한 에이전트를 생성합니다.

In [10]:
# OpenAI 키를 사용하는 경우 gpt-4.1-mini, gpt-4.1 등으로 변경하세요
model = init_chat_model("claude-sonnet-4-5")

agent = create_agent(
    model,
    system_prompt="You are a helpful assistant. Be concise and accurate.",
)

아래는 설정된 시스템 프롬프트를 사용한 에이전트의 응답 예시입니다.

In [11]:
stream_graph(
    agent,
    inputs={"messages": [HumanMessage(content="대한민국의 수도는 어디야?")]},
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
대한민국의 수도는 서울입니다.

### 동적 시스템 프롬프트 (Dynamic Prompting)

런타임 컨텍스트나 에이전트 상태를 기반으로 시스템 프롬프트를 동적으로 생성해야 하는 경우가 있습니다. `dynamic_prompt` 데코레이터를 사용하면 요청마다 다른 시스템 프롬프트를 적용할 수 있습니다.

이 기능은 사용자 역할, 언어 설정, 응답 형식 등을 런타임에 결정해야 할 때 유용합니다. `context_schema`를 정의하여 에이전트 호출 시 필요한 컨텍스트 정보를 전달할 수 있습니다.

아래 코드는 답변 형식과 길이를 동적으로 설정하는 에이전트를 생성합니다.

In [12]:
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    prompt_type: str
    length: int


@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """사용자 역할에 따라 시스템 프롬프트 생성"""
    # 답변 형식 설정
    answer_type = (
        request.runtime.context.get("prompt_type", "default")
        if request.runtime.context
        else "default"
    )
    # 답변 길이 설정
    answer_length = (
        request.runtime.context.get("length", 20) if request.runtime.context else 20
    )
    base_prompt = "You are a helpful assistant. Answer in Korean.\n"

    # 답변 형식에 따라 시스템 프롬프트 생성(동적 프롬프팅)
    if answer_type == "default":
        return f"{base_prompt} [답변 형식] 간결하게 답변해줘. 답변 길이는 {answer_length}자 이하로 해줘."
    elif answer_type == "sns":
        return f"{base_prompt} [답변 형식] SNS 형식으로 답변해줘. 답변 길이는 {answer_length}자 이하로 해줘."
    elif answer_type == "article":
        return f"{base_prompt} [답변 형식] 뉴스 기사 형식으로 답변해줘. 답변 길이는 {answer_length}자 이하로 해줘."
    else:
        return f"{base_prompt} [답변 형식] 간결하게 답변해줘. 답변 길이는 {answer_length}자 이하로 해줘."


# 컨텍스트 스키마와 user_role_prompt 미들웨어를 사용하여 에이전트 생성
agent = create_agent(
    model=model,
    middleware=[user_role_prompt],
    context_schema=Context,
)

In [13]:
# 컨텍스트에 따라 시스템 프롬프트가 동적으로 설정됩니다
stream_graph(
    agent,
    inputs={
        "messages": [HumanMessage(content="머신러닝의 동작 원리에 대해서 설명해줘")]
    },
    context=Context(prompt_type="article", length=1000),
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
# 머신러닝, 컴퓨터가 스스로 학습하는 원리 파헤치기

**[IT/기술]** 인공지능 시대의 핵심 기술로 자리잡은 머신러닝의 작동 메커니즘이 주목받고 있다.

머신러닝(Machine Learning)은 컴퓨터가 명시적인 프로그래밍 없이 데이터로부터 패턴을 학습하여 스스로 판단하고 예측하는 인공지능 기술이다. 전통적인 프로그래밍이 규칙을 직접 입력하는 방식이라면, 머신러닝은 데이터에서 규칙을 찾아내는 방식이다.

## 3단계 학습 과정

머신러닝의 동작 원리는 크게 세 단계로 나뉜다. 첫째, **데이터 수집 및 전처리** 단계에서는 학습에 필요한 대량의 데이터를 모으고 정제한다. 둘째, **모델 학습** 단계에서는 알고리즘이 데이터의 패턴을 분석하고 수학적 모델을 구축한다. 셋째, **예측 및 평가** 단계에서는 학습된 모델로 새로운 데이터를 판단하고 정확도를 측정한다.

## 핵심은 최적화

학습 과정의 핵심은 '오차 최소화'다. 모델은 예측값과 실제값의 차이(오차)를 계산하고, 이를 줄이는 방향으로 내부 파라미터를 반복적으로 조정한다. 이 과정을 '경사하강법'이라 부르며, 수천에서 수만 번 반복하면서 최적의 예측 모델을 만들어낸다.

## 세 가지 학습 방식

머신러닝은 학습 방법에 따라 구분된다. **지도학습**은 정답이 있는 데이터로 학습하며, 이메일 스팸 분류나 질병 진단에 활용된다. **비지도학습**은 정답 없이 데이터의 구조를 파악하며, 고객 세분화에 사용된다. **강화학습**은 시행착오를 통해 보상을 최대화하는 방법을 학습하며, 게임 AI나 자율주행에 적용된다.

업계 전문가들은 "머신러닝은 데이터가 많을수록, 학습 시간이 길수록 성능이 향상되는 특징이 있다"며 "양질의 데이터 확보가 성공의 열쇠"라고 강조했다.

In [14]:
stream_graph(
    agent,
    inputs={
        "messages": [HumanMessage(content="머신러닝의 동작 원리에 대해서 설명해줘")]
    },
    context=Context(prompt_type="sns", length=50),
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
🤖 머신러닝은 데이터로 학습해요!

📊 데이터 → 패턴 발견 → 예측 모델 생성

반복 학습으로 정확도 향상! 🎯

#AI #머신러닝

---

## 미들웨어

미들웨어를 사용하면 모델 호출 전후에 커스텀 로직을 실행할 수 있습니다. `@before_model` 및 `@after_model` 데코레이터를 사용하여 모델 호출을 감싸는 훅을 정의할 수 있습니다.

**미들웨어 활용 사례:**
- 모델 호출 전 메시지 전처리 (예: 쿼리 재작성)
- 모델 호출 후 응답 후처리 (예: 필터링, 로깅)
- 상태 기반 동적 라우팅

아래 코드는 모델 호출 전후에 로깅을 수행하는 미들웨어 예시입니다.

In [15]:
from langchain.agents.middleware import (
    before_model,
    after_model,
)
from langchain.agents.middleware import (
    AgentState,
    ModelRequest,
    ModelResponse,
)
from langchain_core.prompts import PromptTemplate
from langgraph.runtime import Runtime
from typing import Any, Callable


# 노드 스타일: 모델 호출 전 로깅
@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(
        f"\033[95m\n\n모델 호출 전 메시지 {len(state['messages'])}개가 있습니다\033[0m"
    )
    last_message = state["messages"][-1].content
    # OpenAI 키를 사용하는 경우 gpt-4.1-mini, gpt-4.1 등으로 변경하세요
    llm = init_chat_model("claude-sonnet-4-5")

    query_rewrite = (
        PromptTemplate.from_template(
            "Rewrite the following query to be more understandable. Do not change the original meaning. Make it one sentence: {query}"
        )
        | llm
    )
    rewritten_query = query_rewrite.invoke({"query": last_message})

    return {"messages": [rewritten_query.content]}


@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:

    print(
        f"\033[95m\n\n모델 호출 후 메시지 {len(state['messages'])}개가 있습니다\033[0m"
    )
    for i, message in enumerate(state["messages"]):
        print(f"[{i}] {message.content}")
    return None

In [16]:
agent = create_agent(
    model,
    middleware=[
        log_before_model,
        log_after_model,
    ],
)

In [17]:
stream_graph(
    agent,
    inputs={"messages": [HumanMessage(content="대한민국 수도")]},
)



모델 호출 전 메시지 1개가 있습니다

🔄 Node: log_before_model.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
What is the capital city of South Korea?
🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
# Capital of South Korea

The capital city of South Korea is **Seoul** (서울).

## Key Facts about Seoul:
- **Population**: Approximately 9.7 million in the city proper, with over 25 million in the greater metropolitan area
- **Location**: Northwestern South Korea, along the Han River
- **Status**: Largest city in South Korea and the political, economic, and cultural center
- **History**: Has served as Korea's capital for over 600 years, since the Joseon Dynasty (1394)

Seoul is a major global city known for its modern technology, K-pop culture, historical palaces, and vibrant urban life.

모델 호출 후 메시지 3개가 있습니다
[0] 대한민국 수도
[1] What is the capital city of South Korea?
[2] # Capital of South Korea

The capital city of South Korea is **Seoul** (서울).

## Key Facts about Seo

### 클래스 기반 미들웨어

데코레이터 대신 클래스 기반 미들웨어를 사용할 수 있습니다. `AgentMiddleware` 클래스를 상속하고 `before_model` 및 `after_model` 메서드를 오버라이드하여 커스텀 로직을 구현합니다.

클래스 기반 미들웨어는 커스텀 상태 스키마를 정의하거나 복잡한 미들웨어 로직을 구조화할 때 유용합니다.

아래 코드는 클래스 기반 미들웨어를 사용하여 커스텀 상태를 관리하는 예시입니다.

In [18]:
from typing import Any
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware


# 커스텀 상태 스키마 정의
class CustomState(AgentState):
    user_preferences: dict


class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    tools = []

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        # 모델 호출 전 커스텀 로직
        pass


agent = create_agent(model, tools=[], middleware=[CustomMiddleware()])

# 에이전트는 이제 메시지 외에 추가 상태를 추적할 수 있습니다
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "I prefer technical explanations"}],
        "user_preferences": {"style": "technical", "verbosity": "detailed"},
    }
)

### 모델 오류 시 재시도 로직

`wrap_model_call` 데코레이터를 사용하면 모델 호출 실패 시 자동으로 재시도하는 로직을 구현할 수 있습니다. 이는 네트워크 오류나 일시적인 API 장애에 대응하는 데 유용합니다.

아래 코드는 최대 3회까지 재시도하는 미들웨어 예시입니다.

In [19]:
@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"오류 발생으로 {attempt + 1}/3 번째 재시도합니다: {e}")

In [20]:
# OpenAI 키를 사용하는 경우 gpt-4.1-mini, gpt-4.1 등으로 변경하세요
# 일부러 존재하지 않는 모델명을 사용하여 재시도 로직을 테스트합니다
model = init_chat_model("claude-sonnet-4-5-invalid")

agent = create_agent(
    model,
    middleware=[retry_model],
)

In [21]:
stream_graph(
    agent,
    inputs={"messages": [HumanMessage(content="대한민국의 수도는?")]},
)

오류 발생으로 1/3 번째 재시도합니다: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-sonnet-4-5-invalid'}, 'request_id': 'req_011CX8LweDEK9NgLt9dhexUc'}
오류 발생으로 2/3 번째 재시도합니다: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-sonnet-4-5-invalid'}, 'request_id': 'req_011CX8LwfW7bVThHqr49DugM'}


NotFoundError: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-sonnet-4-5-invalid'}, 'request_id': 'req_011CX8LwhFYHFhGVh4wmrHXi'}